# Calling Gemini from Python
In this notebook you will:
1. Paste your API key in **one place**
2. Choose a model (we use the smallest, cheapest one)
3. Set a **system prompt**, a **user prompt** and a **max output size**
4. Check how many tokens your prompt uses (context size)
5. Call the model and read the reply plus token usage

## 0. Add your API key
Get a free key at https://aistudio.google.com/apikey, then paste it in the cell below.
**Never share your key or upload it to GitHub.**

In [ ]:
# ==========================================
#  >>> PASTE YOUR API KEY BETWEEN THE QUOTES <<<
# ==========================================
API_KEY = "PASTE_YOUR_API_KEY_HERE"

In [ ]:
from google import genai
from google.genai import types

if not API_KEY or API_KEY == "PASTE_YOUR_API_KEY_HERE":
    raise RuntimeError("Paste your API key into the API_KEY cell above first.")

client = genai.Client(api_key=API_KEY)
print("Client ready")

## 1. Pick a model
`gemini-2.5-flash-lite` is Google's smallest and most cost-effective Gemini model. Run the next cell to see other "lite" models you could try.

In [ ]:
for m in client.models.list():
    if "lite" in m.name and "tts" not in m.name and "image" not in m.name:
        print(m.name)

In [ ]:
MODEL = "gemini-2.5-flash-lite"  # change me to try another model

## 2. Prompts and size settings
- **System prompt**: sets the model's role and rules.
- **User prompt**: the actual question or task.
- **Context size**: the model's *context window* (how much it can read) is fixed by the model (about 1M tokens here). You control `MAX_OUTPUT_TOKENS`, which caps the length of the reply.
- **Temperature**: 0 = focused and repeatable, higher = more creative.

In [ ]:
SYSTEM_PROMPT = "You are a friendly teacher who explains ideas to beginners in simple words."
USER_PROMPT = "What is a large language model? Explain in 3 short sentences."
MAX_OUTPUT_TOKENS = 300
TEMPERATURE = 0.3

## 3. Check context size
A *token* is roughly 3-4 characters. Let's see how much of the context window your prompt uses.

In [ ]:
info = client.models.get(model=MODEL)
count = client.models.count_tokens(model=MODEL, contents=[SYSTEM_PROMPT, USER_PROMPT])
print(f"Model context window : {info.input_token_limit:,} tokens")
print(f"Max output allowed   : {info.output_token_limit:,} tokens")
print(f"Your prompt          : {count.total_tokens:,} tokens")
print(f"Your output cap      : {MAX_OUTPUT_TOKENS:,} tokens")

## 4. Call the model

In [ ]:
response = client.models.generate_content(
    model=MODEL,
    contents=USER_PROMPT,
    config=types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        max_output_tokens=MAX_OUTPUT_TOKENS,
        temperature=TEMPERATURE,
    ),
)
print(response.text)

In [ ]:
u = response.usage_metadata
print(f"Prompt tokens   : {u.prompt_token_count}")
print(f"Response tokens : {u.candidates_token_count}")
print(f"Total tokens    : {u.total_token_count}")

## 5. The API has no memory
Every call is independent: the model only sees what you send *in that call*. Let's prove it.

In [ ]:
r1 = client.models.generate_content(model=MODEL, contents="My name is Sam.")
print("Call 1:", r1.text)

r2 = client.models.generate_content(model=MODEL, contents="What is my name?")
print("Call 2:", r2.text)  # it has no idea: call 1 was never sent

## 6. Give it memory with a history list
The trick: keep a plain Python **list**. For every turn:
1. append the user's message,
2. send the **whole list** to the model,
3. append the model's reply.

In [ ]:
history = []  # run this cell again to wipe the memory

def chat(user_text):
    history.append(types.Content(role="user", parts=[types.Part(text=user_text)]))
    response = client.models.generate_content(
        model=MODEL,
        contents=history,  # the whole conversation so far
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            max_output_tokens=MAX_OUTPUT_TOKENS,
            temperature=TEMPERATURE,
        ),
    )
    history.append(types.Content(role="model", parts=[types.Part(text=response.text)]))
    print(f"User : {user_text}")
    print(f"Model: {response.text}")
    print(f"(prompt tokens sent: {response.usage_metadata.prompt_token_count})")
    return response.text

In [ ]:
chat("My name is Sam.")
print()
chat("What is my name?")  # now it remembers

In [ ]:
for item in history:
    print(f"{item.role:5}: {item.parts[0].text}")

Notice that the **prompt tokens grow every turn**: the whole history is sent again each time, so long chats cost more and slowly fill the context window.
Keep chatting with `chat("...")`, or run `history = []` to make the model forget.

## Try it yourself
- Change the system prompt to a pirate, a lawyer or a poet and re-run section 4.
- Set `MAX_OUTPUT_TOKENS = 20`. What happens to the answer?
- Set `TEMPERATURE = 1.5` and run several times. What changes?
- Swap `MODEL` for another model from the list and compare token usage.